In [1]:

from google.colab import files
import zipfile
import os

print("Please upload your pakistan_law_dataset.zip file...")
uploaded = files.upload()

# Extract zip
zip_filename = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(f"Dataset extracted successfully!")
print(f"Contents: {os.listdir('/content/pakistan_law_dataset')}")

Please upload your pakistan_law_dataset.zip file...


Saving pakistan_law_dataset.zip to pakistan_law_dataset.zip
Dataset extracted successfully!
Contents: ['csv', 'sqlite', 'xml', 'docs', 'json']


In [2]:

!pip install -q sentence-transformers faiss-cpu google-generativeai pandas tqdm

print("All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.9 MB/s eta 0:00:00
All libraries installed!


In [3]:

import pandas as pd
import json
import os

BASE_DIR = "/content/pakistan_law_dataset"


print("Loading CSV files...")
laws_catalogue = pd.read_csv(f"{BASE_DIR}/csv/laws_catalogue.csv")
sections_all = pd.read_csv(f"{BASE_DIR}/csv/sections_all.csv")
agencies_contacts = pd.read_csv(f"{BASE_DIR}/csv/agencies_contacts.csv")
offense_to_section = pd.read_csv(f"{BASE_DIR}/csv/offense_to_section.csv")

print(f"Loaded:")
print(f" -> Laws: {len(laws_catalogue)}")
print(f" -> Sections: {len(sections_all)}")
print(f" -> Agencies: {len(agencies_contacts)}")
print(f" -> Offense Mappings: {len(offense_to_section)}")




Loading CSV files...
Loaded:
 -> Laws: 1030
 -> Sections: 28249
 -> Agencies: 14
 -> Offense Mappings: 26


In [4]:

laws_catalogue.head(3)

,id,title,short_name,year,status,issued_date,in_force_date,source_url,description,num_provisions,source_file
0,pk-abandoned-properties-management-act-1975,"Abandoned Properties (Management) Act, 1975",APA,1975.0,in_force,1975-02-16,1975-02-16,https://pakistancode.gov.pk/english/UY2FqaJw1-...,Official consolidated text of Abandoned Proper...,31,0001-abandoned-properties-management-act-1975....
1,pk-abolition-of-the-discretionary-quotas-in-ho...,Abolition of the Discretionary Quotas in Housi...,AOTDQ,2013.0,in_force,2013-04-28,2013-04-28,https://pakistancode.gov.pk/english/UY2FqaJw1-...,Official consolidated text of Abolition of the...,4,0002-abolition-of-the-discretionary-quotas-in-...
2,pk-abolition-of-the-punishment-of-whipping-act...,"Abolition of the Punishment of Whipping Act, 1996",AOTPO,1996.0,in_force,1996-11-30,1996-11-30,https://pakistancode.gov.pk/english/UY2FqaJw1-...,Official consolidated text of Abolition of the...,4,0003-abolition-of-the-punishment-of-whipping-a...


In [5]:

from tqdm import tqdm

print("Preparing chunks for embedding...")


chunks = []
for idx, row in tqdm(sections_all.iterrows(), total=len(sections_all), desc="Processing sections"):
    # Combine law info + section content
    chunk_text = f"""
    Law: {row['law_title']}
    Section: {row['section_number']}
    Title: {row['title']}
    Content: {row['content']}
    Penalty: {row.get('punishment_text', 'N/A')}
    """.strip()

    chunks.append({
        "text": chunk_text,
        "law_id": row['law_id'],
        "law_title": row['law_title'],
        "section_number": row['section_number'],
        "title": row['title'],
        "punishment_text": row.get('punishment_text', ''),
        "content": row['content']
    })

print(f"\nCreated {len(chunks)} chunks")
print(f"\nSample chunk:")
print(chunks[0]['text'][:300])

Preparing chunks for embedding...


Processing sections: 100%|██████████| 28249/28249 [00:02<00:00, 13654.72it/s]


Created 28249 chunks

Sample chunk:
Law: Abandoned Properties (Management) Act, 1975
    Section: 1
    Title: Section 1. Short title, extent and commencement
    Content: (1) This act may be called the Abandoned Properties (1[* * *] Management) Act, 1975. (2) It extends to the whole of Pakistan. (3) It shall come into force at once.



In [8]:

from sentence_transformers import SentenceTransformer
import numpy as np

print(" Loading SentenceTransformer model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Model Dimension: {model.get_sentence_embedding_dimension()}")


texts = [chunk['text'] for chunk in chunks]
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

print(f"Embeddings  Shape: {embeddings.shape}")



 Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Dimension: 384


/tmp/ipykernel_4391/3203758548.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Dimension: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/883 [00:00<?, ?it/s]

Embeddings  Shape: (28249, 384)


In [9]:
# Save embeddings for later use
np.save('/content/embeddings.npy', embeddings)
print("Embeddings saved to /content/embeddings.npy")

Embeddings saved to /content/embeddings.npy


In [10]:

import faiss

print("Build FAISS index...")


dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)


index.add(embeddings.astype('float32'))

print(f"FAISS index built with {index.ntotal} vectors")



Build FAISS index...
FAISS index built with 28249 vectors


In [11]:

faiss.write_index(index, '/content/faiss_law_index.index')


In [12]:

def search_laws(query, top_k=5):

    query_embedding = model.encode([query], convert_to_numpy=True).astype('float32')

    distances, indices = index.search(query_embedding, k=top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "chunk": chunks[idx],
            "similarity": 1 - (distances[0][i] / 2),
            "distance": distances[0][i]
        })

    return results


test_query = "cyber harassment punishment"
print(f"🔍 Testing search: '{test_query}'")
results = search_laws(test_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"\n{i}. {r['chunk']['law_title']} - Section {r['chunk']['section_number']}")
    print(f"   Similarity: {r['similarity']:.2%}")
    print(f"   {r['chunk']['content'][:200]}...")

🔍 Testing search: 'cyber harassment punishment'

1. Prevention of Electronic Crimes Act, 2016 - Section 24
   Similarity: 65.37%
   (1) A person commits the offence of cyber stalking who, with the intent to coerce or intimidate or harass any person, uses information system, information system network, the Internet, website, electr...

2. Prevention of Electronic Crimes Act, 2016 - Section 11
   Similarity: 56.19%
   Whoever prepares or disseminates information, through any information system or device, that advances or is likely to advance interfaith, sectarian or racial hatred, shall be punished with imprisonmen...

3. Prevention of Electronic Crimes Act, 2016 - Section 10
   Similarity: 53.22%
   Whoever commits or threatens to commit any of the offences under sections 6, 7, 8 or 9, where the commission or threat is with the intent to,— (a)     coerce, intimidate, create a sense of fear, panic...
